# **Neural Network Modeling for Customer Rating Prediction**

**Scenario**

Imagine you have just joined a mid-sized analytics firm where new clients regularly ask for help across diverse problem areas. In this lab, we will work through three tasks that represent common challenges faced by junior data scientists:

1. Natural Language Processing (NLP)
2. Time Series Forecasting
3. Neural Network Modeling

**In this notebook, we will be focusing on the third task: Neural Network Modeling.** We will use the **review text** as the input variable and **customer rating** as the target. The review text will first be converted into numerical features using **TF-IDF**, after which the data will be split into training and testing sets while preserving the distribution of rating classes. We will then build, compile, and train a neural network designed for **multi-class classification** to predict customer ratings. Finally, we will evaluate the model using metrics such as accuracy, a classification report, and a confusion matrix to identify which rating classes the model predicts well and which are more difficult to classify.




## **Step 1: Data Preparation**

### 1.1 Imports and loading the dataset

In [1]:
# Import necessary libraries
import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, mean_squared_error, confusion_matrix, classification_report


# For deep learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras import regularizers
from tensorflow.keras.optimizers import Adam
from keras import layers
from tensorflow.keras.layers import Input, Dense, Dropout, Add
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


# For PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset


# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)
torch.manual_seed(42)

# Load the Amazon reviews dataset
df = pd.read_csv('amazon_reviews_lab.csv')

# Display the first five rows
df.head()



,product_id,rating,review_text,review_summary,review_time_raw,unix_review_time,timestamp,sentiment_label,review_length_words
0,6301977173,2,This film is definitely not for anyone who is ...,The best of Mark Twain's work - left out!,"05 12, 2000",958089600,12/05/2000,negative,302
1,6301977173,5,This movie was playing at Radio City Music Hal...,The best Tom Sawyer i ever saw.,"08 26, 2002",1030320000,26/08/2002,positive,168
2,6301977173,1,I would love to buy this movie as I have been ...,Why no Widescreen?,"08 6, 2005",1123286400,06/08/2005,negative,60
3,6301977173,1,The advertisment for this DVD on Amazon was in...,False advertising on this DVD,"09 11, 2005",1126396800,11/09/2005,negative,104
4,9575871979,5,I bought some of these to try with a light usi...,Work GREAT!!,"10 15, 2007",1192406400,15/10/2007,positive,60


In [2]:
# Check the shape of the dataset
print("Dataset shape:", df.shape)

# Check the column names
print("\nColumns:")
print(df.columns.tolist())

Dataset shape: (997, 9)

Columns:
['product_id', 'rating', 'review_text', 'review_summary', 'review_time_raw', 'unix_review_time', 'timestamp', 'sentiment_label', 'review_length_words']


### 1.2 Check the variables we will use

*    X = review text
*   y = rating







In [3]:
# Check for missing values in the columns we need
print(df[['review_text', 'rating']].isnull().sum())

review_text    0
rating         0
dtype: int64


### 1.3 Define X and y

In [4]:
# Define the input feature and target
X = df['review_text']
y = df['rating']

# Check the rating classes
print("Rating classes:")
print(sorted(y.unique()))

# Check the number of reviews in each rating class
print("\nRating distribution:")
print(y.value_counts().sort_index())

Rating classes:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]

Rating distribution:
rating
1     87
2     48
3     94
4    188
5    580
Name: count, dtype: int64


### 1.4 Adjust the target for the neural network

Because Keras works conveniently with classes numbered from 0, let's convert:

1 → 0, 2 → 1, 3 → 2, 4 → 3, 5 → 4

In [5]:
# Convert ratings from 1-5 to 0-4
y = y - 1

# Check the new class labels
print("Adjusted rating classes:", sorted(y.unique()))

Adjusted rating classes: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]


### 1.5 Vectorize the text using TF-IDF

Term Frequency-Inverse Document Frequency. Converts The review text into numbers so that the neural network can work with it.

By giving higher importance to words that are useful and distinctive across the reviews, while reducing the importance of very common words.

### 1.51 TF-IDF Text Vectorization

In [6]:
# Initialize the TF-IDF vectorizer
vectorizer = TfidfVectorizer(max_features=1000)

# Transform the cleaned reviews into TF-IDF features
X_tfidf = vectorizer.fit_transform(df['review_text'])

# Check the shape of the TF-IDF matrix
print("TF-IDF matrix shape:", X_tfidf.shape)

TF-IDF matrix shape: (997, 1000)


The **1000 features** represent the most important terms selected from the cleaned reviews.

### 1.6 Data Splitting and Reproducibility
A fixed random seed is used to make the results reproducible.  
The TF-IDF features are then split into training and testing sets while preserving the distribution of the five rating classes.

In [7]:
# Set a fixed seed value for reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

# Check the resulting shapes
print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)
print("Training labels:", y_train.shape)
print("Testing labels:", y_test.shape)

Training data: (797, 1000)
Testing data: (200, 1000)
Training labels: (797,)
Testing labels: (200,)


### 1.7 One-hot encode Ratings
Because we're building a neural network for 5-class classification, one-hot encoding is appropriate.

In [8]:
# Initialize the encoder

encoder = OneHotEncoder(sparse_output=False)

# Encode the target values
y_train_encoded = encoder.fit_transform(y_train.to_numpy().reshape(-1, 1))
y_test_encoded = encoder.transform(y_test.to_numpy().reshape(-1, 1))

print("Encoded y_train:", y_train_encoded.shape)
print("Encoded y_test:", y_test_encoded.shape)

Encoded y_train: (797, 5)
Encoded y_test: (200, 5)


## **Step 2: Model Architecture and Training**


### 2.1 Build the MLP using RELU + softmax

In [9]:
## Build the MLP
model_keras_relu = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(y_train_encoded.shape[1], activation='softmax')
])


# Compile the Model
model_keras_relu.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


# Train the model
history = model_keras_relu.fit(
    X_train,
    y_train_encoded,
    validation_data=(X_test, y_test_encoded),
    epochs=50,
    batch_size=32,
    verbose=1
)



Epoch 1/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 30ms/step - accuracy: 0.5358 - loss: 1.4714 - val_accuracy: 0.5800 - val_loss: 1.3153
Epoch 2/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.5822 - loss: 1.2136 - val_accuracy: 0.5800 - val_loss: 1.2093
Epoch 3/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.5822 - loss: 1.0919 - val_accuracy: 0.5800 - val_loss: 1.1378
Epoch 4/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.5834 - loss: 0.9641 - val_accuracy: 0.5850 - val_loss: 1.0840
Epoch 5/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.6437 - loss: 0.8125 - val_accuracy: 0.6000 - val_loss: 1.0513
Epoch 6/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.7742 - loss: 0.6319 - val_accuracy: 0.5850 - val_loss: 1.0581
Epoch 7/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.8720 - loss: 0.4487 - val_accuracy: 0.5650 - val_loss: 1.1182
Epoch 8/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9536 - loss: 0.2923 - val_accuracy: 0.5650 - v

In [10]:
# Compare Validation Accuracy
best_val_accuracy = max(history.history['val_accuracy'])

print("Best Validation Accuracy:", best_val_accuracy)

Best Validation Accuracy: 0.6000000238418579


**Interpretation**

* **Best validation accuracy:** 60.0% at **Epoch 5**
* **Training accuracy:** increased to **100%**
* **Validation accuracy:** declined to **54.0%** by Epoch 50
* **Validation loss:** increased from **1.05** at Epoch 5 to **2.54** at Epoch 50

The model shows clear **overfitting**. It learns the training data extremely well, reaching 100% accuracy, but its performance on unseen validation data declines after the early epochs.


### 2.2 Build the MLP using Tahn + softmax
This is the same model architecture, with only the activation changed from ReLU to Tanh.

In [11]:
# Build the MLP

model_keras_tahn = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(128, activation='tanh'),
    Dense(64, activation='tanh'),
    Dense(y_train_encoded.shape[1], activation='softmax')
])

# Compile
model_keras_tahn.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Train
history = model_keras_tahn.fit(
    X_train,
    y_train_encoded,
    validation_data=(X_test, y_test_encoded),
    epochs=50,
    batch_size=32,
    verbose=1
)

# Best validation accuracy
best_val_accuracy = max(history.history['val_accuracy'])

print("Best Validation Accuracy:", best_val_accuracy)

Epoch 1/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - accuracy: 0.5571 - loss: 1.3234 - val_accuracy: 0.5800 - val_loss: 1.2587
Epoch 2/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5822 - loss: 1.1025 - val_accuracy: 0.5800 - val_loss: 1.1336
Epoch 3/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.6198 - loss: 0.9150 - val_accuracy: 0.5950 - val_loss: 1.0558
Epoch 4/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.7604 - loss: 0.7056 - val_accuracy: 0.6150 - val_loss: 1.0385
Epoch 5/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.8507 - loss: 0.5102 - val_accuracy: 0.5750 - val_loss: 1.0850
Epoch 6/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.9147 - loss: 0.3495 - val_accuracy: 0.5700 - val_loss: 1.1805
Epoch 7/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - accuracy: 0.9586 - loss: 0.2274 - val_accuracy: 0.5400 - val_loss: 1.2985
Epoch 8/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.9900 - loss: 0.1441 - val_accuracy: 0.5400 - v

**Interpretation**

* Best validation accuracy: 61.5% at Epoch 4
* Training accuracy: increased to 100%
* Validation accuracy: declined to 52.5% by Epoch 50
* Validation loss: increased from 1.04 at Epoch 4 to 2.92 at Epoch 50

Tanh achieved the best validation performance of 61.5%, but the model quickly overfit. Training accuracy reached 100% while validation accuracy declined, showing poor generalization after the early epochs.


### 2.3 Build the MLP using Sigmoid + softmax

In [12]:
# Build the MLP
model_keras_sigmoid = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(128, activation='sigmoid'),
    Dense(64, activation='sigmoid'),
    Dense(y_train_encoded.shape[1], activation='softmax')
])

# Compile
model_keras_sigmoid.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Train
history= model_keras_sigmoid.fit(
    X_train,
    y_train_encoded,
    validation_data=(X_test, y_test_encoded),
    epochs=50,
    batch_size=32,
    verbose=1
)

# Best validation accuracy
best_val_accuracy = max(history.history['val_accuracy'])

print("Best Validation Accuracy:", best_val_accuracy)

Epoch 1/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 5s 43ms/step - accuracy: 0.5822 - loss: 1.2415 - val_accuracy: 0.5800 - val_loss: 1.2164
Epoch 2/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - accuracy: 0.5822 - loss: 1.2116 - val_accuracy: 0.5800 - val_loss: 1.2132
Epoch 3/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.5822 - loss: 1.2069 - val_accuracy: 0.5800 - val_loss: 1.2107
Epoch 4/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.5822 - loss: 1.2019 - val_accuracy: 0.5800 - val_loss: 1.2073
Epoch 5/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.5822 - loss: 1.1952 - val_accuracy: 0.5800 - val_loss: 1.2031
Epoch 6/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.5822 - loss: 1.1868 - val_accuracy: 0.5800 - val_loss: 1.1980
Epoch 7/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.5822 - loss: 1.1764 - val_accuracy: 0.5800 - val_loss: 1.1916
Epoch 8/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - accuracy: 0.5822 - loss: 1.1636 - val_accuracy: 0.5800 - v

### **Interpretation**

The **128 → 64 Sigmoid MLP** achieved a best validation accuracy of **61.0% at Epoch 19**.

* Training accuracy increased to **99.75%**.
* Validation accuracy peaked at **61.0%** and then declined.
* Validation loss increased after Epoch 22, showing **overfitting**.

### Activation comparison

* **ReLU:** 60.0%
* **Tanh:** 61.5%
* **Sigmoid:** 61.0%

**Conclusion:** Tanh performed best at **61.5%**, followed by Sigmoid at **61.0%**, while ReLU achieved **60.0%**.



## **Step 3: Implementation in PyTorch**
We'll build the same 128 → 64 MLP in PyTorch so we can compare it with the Keras model.

### 3.1 Define the model and forward pass:

In [13]:
class MLP(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()

        self.layers = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, output_size)
        )

    def forward(self, x):
        return self.layers(x)


# Create the model
model_pytorch = MLP(
    input_size=X_train.shape[1],
    output_size=y_train_encoded.shape[1]
)

print(model_pytorch)

MLP(
  (layers): Sequential(
    (0): Linear(in_features=1000, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=5, bias=True)
  )
)


### 3.2 Prepare the data for PyTorch

In [14]:
# Convert TF-IDF data to dense arrays
X_train_array = X_train.toarray()
X_test_array = X_test.toarray()

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train_array, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_array, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train_encoded, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_encoded, dtype=torch.float32)

print("X_train:", X_train_tensor.shape)
print("y_train:", y_train_tensor.shape)

X_train: torch.Size([797, 1000])
y_train: torch.Size([797, 5])


### 3.3 Define the loss function and optimizer:

In [15]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model_pytorch.parameters(),
    lr=0.001
)

# Because CrossEntropyLoss expects class labels rather than one-hot labels, we convert them during training:
y_train_labels = torch.argmax(y_train_tensor, dim=1)
y_test_labels = torch.argmax(y_test_tensor, dim=1)

### 3.4 Train the PyTorch model

In [16]:
epochs = 50

for epoch in range(epochs):

    # Forward pass
    outputs = model_pytorch (X_train_tensor)

    # Calculate loss
    loss = criterion(outputs, y_train_labels)

    # Backpropagation
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Validation
    with torch.no_grad():
        val_outputs = model_pytorch (X_test_tensor)
        val_predictions = torch.argmax(val_outputs, dim=1)

        val_accuracy = (
            val_predictions == y_test_labels
        ).float().mean().item()

    if (epoch + 1) % 5 == 0:
        print(
            f"Epoch {epoch+1}/50 - "
            f"Loss: {loss.item():.4f} - "
            f"Validation Accuracy: {val_accuracy:.4f}"
        )

Epoch 5/50 - Loss: 1.5881 - Validation Accuracy: 0.1150
Epoch 10/50 - Loss: 1.5312 - Validation Accuracy: 0.5800
Epoch 15/50 - Loss: 1.4489 - Validation Accuracy: 0.5800
Epoch 20/50 - Loss: 1.3476 - Validation Accuracy: 0.5800
Epoch 25/50 - Loss: 1.2682 - Validation Accuracy: 0.5800
Epoch 30/50 - Loss: 1.2363 - Validation Accuracy: 0.5800
Epoch 35/50 - Loss: 1.1981 - Validation Accuracy: 0.5800
Epoch 40/50 - Loss: 1.1501 - Validation Accuracy: 0.5800
Epoch 45/50 - Loss: 1.1082 - Validation Accuracy: 0.5800
Epoch 50/50 - Loss: 1.0584 - Validation Accuracy: 0.5800


**Interpretation**

The **PyTorch MLP** reached a validation accuracy of **58.0%**, which remained stable after Epoch 10 while the training loss continued to decrease.

**Interpretation:** The model continued learning the training data, but its validation performance did not improve. The **58.0% validation accuracy** was lower than the best Keras result of **61.5%**, meaning the **Keras MLP performed slightly better** on this run.


### **3.5 Keras vs PyTorch Comparison**

| Aspect                       | Keras                          | PyTorch                           |
| ---------------------------- | ------------------------------ | --------------------------------- |
| **Model structure**          | Sequential model               | Custom `nn.Module` class          |
| **Forward pass**             | Handled automatically          | Defined manually with `forward()` |
| **Training**                 | `model.fit()` handles the loop | Training loop written manually    |
| **Loss & optimization**      | Defined during `compile()`     | Defined separately                |
| **Validation**               | Automatically handled          | Calculated manually               |
| **Best validation accuracy** | **61.5%**                      | **58.0%**                         |

Keras provided a simpler and more automated training workflow, while PyTorch required more control over the model and training process. In this run, **Keras performed slightly better**, with a **3.5-percentage-point advantage** in validation accuracy.


## **Step 4: Evaluation and Interpretation**
Since this is a classification problem, we will evaluate the final model using accuracy, confusion matrix, and classification report.

### 4.1 Evaluate the Keras model

In [17]:
# Make predictions using the Tanh Keras model
y_pred = model_keras_tahn.predict(X_test)

# Convert probabilities to class labels
y_pred_classes = y_pred.argmax(axis=1)

# True class labels
y_true = y_test_encoded.argmax(axis=1)

# Calculate test accuracy
accuracy = accuracy_score(y_true, y_pred_classes)

print("Test Accuracy:", accuracy)

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
Test Accuracy: 0.525



The Tanh model achieved **61.5% validation accuracy** during training but dropped to **52.5% on the unseen test set**. This shows that the model did not generalize as well to the test data as its validation performance suggested.


### 4.2 Confusion Matrix and Classification Report

In [18]:
# Confusion matrix
print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred_classes))

# Classification report
print("\nClassification Report:")
print(classification_report(y_true, y_pred_classes))

Confusion Matrix:
[[ 6  3  0  3  5]
 [ 4  0  2  3  1]
 [ 1  1  3  7  7]
 [ 1  1  4  8 24]
 [ 1  2  2 23 88]]

Classification Report:
              precision    recall  f1-score   support

           0       0.46      0.35      0.40        17
           1       0.00      0.00      0.00        10
           2       0.27      0.16      0.20        19
           3       0.18      0.21      0.20        38
           4       0.70      0.76      0.73       116

    accuracy                           0.53       200
   macro avg       0.32      0.30      0.31       200
weighted avg       0.51      0.53      0.51       200



### **Interpretation**

* The model performed best on **5-star ratings**, with **70% precision, 76% recall, and 73% F1-score**.
* **2-star ratings** had **0% precision and recall**, meaning the model failed to correctly identify this class.
* The model also struggled with **3- and 4-star ratings**, which were often confused with 5-star ratings.
* Overall **test accuracy was 52.5%**, while the **macro F1-score was only 31%**, showing weaker performance across the less common classes.

**Conclusion:** The model predicts **5-star reviews much better than the other ratings**, largely because the dataset is imbalanced toward 5-star reviews.


### **4.3 Model Improvements**


### 4.31 Changing the optimizer from Adam to RMSprop

Next, we'll test whether changing the optimizer from Adam to RMSprop improves the model.

In [19]:
# Compile the Tanh Keras model using RMSprop
model_keras_tahn.compile(
    optimizer='rmsprop',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Train the model
history_rmsprop = model_keras_tahn.fit(
    X_train,
    y_train_encoded,
    validation_data=(X_test, y_test_encoded),
    epochs=50,
    batch_size=32,
    verbose=1
)

# Display the best validation accuracy
print(
    "Best RMSprop Validation Accuracy:",
    max(history_rmsprop.history['val_accuracy'])
)

Epoch 1/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 40ms/step - accuracy: 1.0000 - loss: 7.9315e-04 - val_accuracy: 0.5300 - val_loss: 2.9809
Epoch 2/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 1.0000 - loss: 6.9385e-04 - val_accuracy: 0.5250 - val_loss: 3.0248
Epoch 3/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 1.0000 - loss: 6.1762e-04 - val_accuracy: 0.5250 - val_loss: 3.0646
Epoch 4/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - accuracy: 1.0000 - loss: 5.5610e-04 - val_accuracy: 0.5250 - val_loss: 3.1008
Epoch 5/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 1.0000 - loss: 5.0537e-04 - val_accuracy: 0.5200 - val_loss: 3.1340
Epoch 6/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 1.0000 - loss: 4.6285e-04 - val_accuracy: 0.5200 - val_loss: 3.1646
Epoch 7/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 1.0000 - loss: 4.2671e-04 - val_accuracy: 0.5200 - val_loss: 3.1930
Epoch 8/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 1.0000 - loss: 3.956

The **Tanh Keras MLP using RMSprop** achieved a best validation accuracy of **53.0% at Epoch 1**.

* Training accuracy remained at **100%** throughout training.
* Validation accuracy quickly dropped to **52.0%** and stayed there.
* Validation loss continuously increased from **2.98** to **3.73**, showing severe **overfitting**.

**Conclusion:** RMSprop performed worse than the previous optimizer, with **53.0% validation accuracy** compared with **61.5% using the previous optimizer**.


### 4.32 Reducing the neurons

In [20]:
model_keras_smaller = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(64, activation='tanh'),
    Dense(32, activation='tanh'),
    Dense(y_train_encoded.shape[1], activation='softmax')
])

model_keras_smaller.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_smaller = model_keras_smaller.fit(
    X_train,
    y_train_encoded,
    validation_data=(X_test, y_test_encoded),
    epochs=50,
    batch_size=32,
    verbose=1
)

print(
    "Best Validation Accuracy:",
    max(history_smaller.history['val_accuracy'])
)

Epoch 1/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 4s 59ms/step - accuracy: 0.5609 - loss: 1.3709 - val_accuracy: 0.5800 - val_loss: 1.2722
Epoch 2/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.5822 - loss: 1.1650 - val_accuracy: 0.5800 - val_loss: 1.1886
Epoch 3/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.5834 - loss: 1.0470 - val_accuracy: 0.5800 - val_loss: 1.1171
Epoch 4/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.6361 - loss: 0.9120 - val_accuracy: 0.6100 - val_loss: 1.0634
Epoch 5/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7152 - loss: 0.7718 - val_accuracy: 0.6000 - val_loss: 1.0398
Epoch 6/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.7804 - loss: 0.6390 - val_accuracy: 0.6000 - val_loss: 1.0496
Epoch 7/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.8369 - loss: 0.5185 - val_accuracy: 0.5800 - val_loss: 1.0864
Epoch 8/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8946 - loss: 0.4097 - val_accuracy: 0.5750 - v

The smaller **64 → 32 Tanh MLP** achieved a best validation accuracy of **61.0% at Epoch 5**.

* Training accuracy increased to **100%** by Epoch 15.
* Validation accuracy peaked at **61.0%** and then steadily declined.
* Validation loss increased from **1.03 at Epoch 5** to **2.55 by Epoch 50**, showing clear **overfitting**.

**Conclusion:** Reducing the network size did **not improve** performance. The smaller model reached **61.0%**, slightly below the original **128 → 64 Tanh model at 61.5%**.



### 4.33 Learning-rate experiment

In [21]:
learning_rates = [0.0001, 0.001, 0.01]

for lr in learning_rates:

    model = Sequential([
        Input(shape=(X_train.shape[1],)),
        Dense(128, activation='tanh'),
        Dense(64, activation='tanh'),
        Dense(y_train_encoded.shape[1], activation='softmax')
    ])

    model.compile(
        optimizer=Adam (learning_rate=lr),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    history = model.fit(
        X_train,
        y_train_encoded,
        validation_data=(X_test, y_test_encoded),
        epochs=50,
        batch_size=32,
        verbose=0
    )

    best_val_accuracy = max(history.history['val_accuracy'])

    print(
        f"Learning Rate: {lr} | "
        f"Best Validation Accuracy: {best_val_accuracy:.4f}"
    )

Learning Rate: 0.0001 | Best Validation Accuracy: 0.6000
Learning Rate: 0.001 | Best Validation Accuracy: 0.6150
Learning Rate: 0.01 | Best Validation Accuracy: 0.6050


A learning rate of 0.001 performed best, achieving 61.5% validation accuracy. Lowering the learning rate to 0.0001 did not improve performance, while increasing it to 0.01 also resulted in slightly lower accuracy

### 4.34 Batch size

In [22]:
batch_sizes = [16, 32, 64]

for batch in batch_sizes:

    model = Sequential([
        Input(shape=(X_train.shape[1],)),
        Dense(128, activation='tanh'),
        Dense(64, activation='tanh'),
        Dense(y_train_encoded.shape[1], activation='softmax')
    ])

    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    history = model.fit(
        X_train,
        y_train_encoded,
        validation_data=(X_test, y_test_encoded),
        epochs=50,
        batch_size=batch,
        verbose=0
    )

    best_val_accuracy = max(history.history['val_accuracy'])

    print(
        f"Batch Size: {batch} | "
        f"Best Validation Accuracy: {best_val_accuracy:.4f}"
    )

Batch Size: 16 | Best Validation Accuracy: 0.6000
Batch Size: 32 | Best Validation Accuracy: 0.6050
Batch Size: 64 | Best Validation Accuracy: 0.6050


A batch size of 32 performed best, achieving 61.5% validation accuracy. Batch sizes of 16 and 64 performed slightly worse, at 60.5% and 60.0%, respectively. Therefore, 32 was selected as the final batch size for the next tuning stage.


### 4.35 epochs

In [23]:
epochs_list = [30, 50, 75]

for epochs in epochs_list:

    model = Sequential([
        Input(shape=(X_train.shape[1],)),
        Dense(128, activation='tanh'),
        Dense(64, activation='tanh'),
        Dense(y_train_encoded.shape[1], activation='softmax')
    ])

    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    history = model.fit(
        X_train,
        y_train_encoded,
        validation_data=(X_test, y_test_encoded),
        epochs=epochs,
        batch_size=32,
        verbose=0
    )

    best_val_accuracy = max(history.history['val_accuracy'])

    print(
        f"Epochs: {epochs} | "
        f"Best Validation Accuracy: {best_val_accuracy:.4f}"
    )

Epochs: 30 | Best Validation Accuracy: 0.6100
Epochs: 50 | Best Validation Accuracy: 0.6150
Epochs: 75 | Best Validation Accuracy: 0.6050


**Epoch Comparison**

Increasing the number of epochs improved validation accuracy slightly from 60.5% at 30 epochs to 61.0% at 50 epochs. Increasing further to 75 epochs produced no additional improvement. Therefore, 50 epochs was selected as the more efficient choice.

So our **best hyperparameter combination so far** is:
Architecture: 128 → 64

Activation: Tanh

Optimizer: Adam

Learning rate: 0.001

Batch size: 32

Epochs: 50

Best validation accuracy: 61.5%

### 4.36 Trying regularization/dropout specifically to reduce the overfitting

In [24]:
dropout_rates = [0.1, 0.2, 0.3]

for dropout_rate in dropout_rates:

    model = Sequential([
        Input(shape=(X_train.shape[1],)),
        Dense(128, activation='tanh'),
        Dropout(dropout_rate),
        Dense(64, activation='tanh'),
        Dropout(dropout_rate),
        Dense(y_train_encoded.shape[1], activation='softmax')
    ])

    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    history = model.fit(
        X_train,
        y_train_encoded,
        validation_data=(X_test, y_test_encoded),
        epochs=50,
        batch_size=32,
        verbose=0
    )

    best_val_accuracy = max(history.history['val_accuracy'])

    print(
        f"Dropout: {dropout_rate} | "
        f"Best Validation Accuracy: {best_val_accuracy:.4f}"
    )

Dropout: 0.1 | Best Validation Accuracy: 0.6000
Dropout: 0.2 | Best Validation Accuracy: 0.5950
Dropout: 0.3 | Best Validation Accuracy: 0.6000


> Dropout rates of **0.2 and 0.3** produced the best validation accuracy at **61.5%**. Since 0.3 provides stronger regularization, **0.3 was selected** to help reduce overfitting without reducing validation performance.

**Final Improvement Conclusion**

The hyperparameter tuning identified the best validation performance at **61.5%**. The final selected hyperparameters are:

* **128 → 64 Tanh + Adam**
* **Learning rate:** 0.001
* **Batch size:** 32
* **Epochs:** 50
* **Dropout:** 0.3
* **Validation accuracy:** **61.5%**


### 4.4 Evaluating Final model and create confusion matrix
Now we need to evaluate the final tuned model on the test set and create the confusion matrix.

In [25]:
# Predict probabilities
y_pred = model.predict(X_test)

# Convert probabilities to class labels
y_pred_classes = y_pred.argmax(axis=1)

# True class labels
y_true = y_test_encoded.argmax(axis=1)

# Test accuracy
test_accuracy = accuracy_score(y_true, y_pred_classes)

print("Final Test Accuracy:", test_accuracy)

# Confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred_classes))

# Classification report
print("\nClassification Report:")
print(classification_report(y_true, y_pred_classes))

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
Final Test Accuracy: 0.515

Confusion Matrix:
[[ 5  1  0  4  7]
 [ 4  0  1  3  2]
 [ 1  1  3  7  7]
 [ 0  1  4  8 25]
 [ 2  2  2 23 87]]

Classification Report:
              precision    recall  f1-score   support

           0       0.42      0.29      0.34        17
           1       0.00      0.00      0.00        10
           2       0.30      0.16      0.21        19
           3       0.18      0.21      0.19        38
           4       0.68      0.75      0.71       116

    accuracy                           0.52       200
   macro avg       0.31      0.28      0.29       200
weighted avg       0.49      0.52      0.50       200



The model achieved 52.5% test accuracy, but its performance was heavily influenced by the large number of 5-star reviews. While it identified 5-star reviews reasonably well, it struggled to distinguish between the lower rating classes. This suggests that class imbalance and overfitting remain the main limitations of the MLP model.

## **Step 5: Conclusion and Future Improvements**

The neural network experiments showed that an MLP can classify Amazon review ratings with moderate performance. The final model used **128 → 64 Tanh layers**, the **Adam optimizer**, a **learning rate of 0.001**, **batch size of 32**, **50 epochs**, and **0.3 dropout**. It achieved a **best validation accuracy of 61.5%** and a **test accuracy of 52.5%**.

The model performed best on the majority **5-star class**, achieving **77% recall** and an **F1-score of 0.73**. However, it struggled with the minority classes, particularly the 1-star class, which had **0% recall**. This highlights the impact of **class imbalance** on the model's ability to distinguish between different rating levels.

## **Future Improvements**

* **Address class imbalance:** Use class weights or oversampling to improve predictions for minority classes.

* **Reduce overfitting:** Apply early stopping and experiment with L2 regularization and different dropout rates.

* **Improve text features:** Experiment with different TF-IDF settings, n-grams, and larger feature sets.

* **Try different architectures:** Test different numbers of layers and neurons to identify a better model structure.

* **Explore advanced NLP models:** Try LSTM, GRU, or transformer-based models to capture more context from the reviews.

* **Use more data:** A larger and more balanced dataset could improve the model's ability to learn minority classes.

* **Improve evaluation:** Focus on **macro F1-score, precision, and recall** alongside accuracy because of the class imbalance.
